In [2]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
RANDOMSEED=1727

os.environ['PYTHONHASHSEED'] = str(RANDOMSEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ":4096:2"

In [3]:
import torch,random
import tensorflow as tf
import numpy as np

def random_seed(seed=RANDOMSEED, use_cuda=False):
    np.random.seed(seed) # cpu vars
    torch.manual_seed(seed) # cpu vars
    random.seed(seed) # Python
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    torch.use_deterministic_algorithms(True)
    # tf.config.threading.set_inter_op_parallelism_threads(1)
    # tf.config.threading.set_intra_op_parallelism_threads(1)
    if use_cuda: 
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # gpu vars
        torch.backends.cudnn.deterministic = True  #needed
        torch.backends.cudnn.benchmark = False
        
random_seed(RANDOMSEED)
tf.config.set_visible_devices([], 'GPU')

In [4]:
from pathlib import Path

cred_path = Path('~/.kaggle/access_token').expanduser()
if not cred_path.exists():
    cred_path.parent.mkdir(exist_ok=True)
    cred_path.write_text("KGAT_9f6b15aaf6f7637b8497dfb3c56c079e")
    cred_path.chmod(0o600)

In [5]:
compName = Path('nlp-getting-started')

In [6]:
if not iskaggle:
    if not compName.exists():
        import zipfile,kaggle
        kaggle.api.competition_download_cli(str(compName))
        zipfile.ZipFile(f'{compName}.zip').extractall(compName)
else:
    # /kaggle/input/competitions/nlp-getting-started/train.csv
    compName = Path(f'/kaggle/input/competitions/{compName}')

# %pip install -q datasets
!dir /o:g /w {compName}
# !ls {compName}

 Volume in drive C is Windows
 Volume Serial Number is 6291-898F

 Directory of c:\Users\longnuub\learning-programming-languages\learning-python\kaggle\nlp-getting-started

[..]                    [.]                     test.csv
train.csv               sample_submission.csv   
               3 File(s)      1.431.241 bytes
               2 Dir(s)  137.348.968.448 bytes free


In [7]:
import pandas as pd
train=pd.read_csv(compName/"train.csv")
test=pd.read_csv(compName/"test.csv")

# drop the `id` col for training data ONLY, we need the id for test preds later
train.drop(columns=["id"],inplace=True)
train[27:41]

,keyword,location,text,target
27,NaN,NaN,Love my girlfriend,0
28,NaN,NaN,Cooool :),0
29,NaN,NaN,Do you like pasta?,0
30,NaN,NaN,The end!,0
31,ablaze,Birmingham,@bbcmtd Wholesale Markets ablaze http://t.co/l...,1
32,ablaze,Est. September 2012 - Bristol,We always try to bring the heavy. #metal #RT h...,0
33,ablaze,AFRICA,#AFRICANBAZE: Breaking news:Nigeria flag set a...,1
34,ablaze,"Philadelphia, PA",Crying out for more! Set me ablaze,0
35,ablaze,"London, UK",On plus side LOOK AT THE SKY LAST NIGHT IT WAS...,0
36,ablaze,Pretoria,@PhDSquares #mufc they've built so much hype a...,0


# feat engineering

In [8]:
train["keywordDefined"]=pd.Series([
  int(pd.notna(keyword)) for keyword in train["keyword"]
])
train["locationDefined"]=pd.Series([
  int(pd.notna(loc)) for loc in train["location"]
])
train[27:41]

,keyword,location,text,target,keywordDefined,locationDefined
27,NaN,NaN,Love my girlfriend,0,0,0
28,NaN,NaN,Cooool :),0,0,0
29,NaN,NaN,Do you like pasta?,0,0,0
30,NaN,NaN,The end!,0,0,0
31,ablaze,Birmingham,@bbcmtd Wholesale Markets ablaze http://t.co/l...,1,1,1
32,ablaze,Est. September 2012 - Bristol,We always try to bring the heavy. #metal #RT h...,0,1,1
33,ablaze,AFRICA,#AFRICANBAZE: Breaking news:Nigeria flag set a...,1,1,1
34,ablaze,"Philadelphia, PA",Crying out for more! Set me ablaze,0,1,1
35,ablaze,"London, UK",On plus side LOOK AT THE SKY LAST NIGHT IT WAS...,0,1,1
36,ablaze,Pretoria,@PhDSquares #mufc they've built so much hype a...,0,1,1


In [9]:
train["keywordInText"]=pd.Series([
	int(str(keyword).lower() in str(text).lower()) if keyword else False for keyword,text in zip(train["keyword"],train["text"])
])
train[27:41]

,keyword,location,text,target,keywordDefined,locationDefined,keywordInText
27,NaN,NaN,Love my girlfriend,0,0,0,0
28,NaN,NaN,Cooool :),0,0,0,0
29,NaN,NaN,Do you like pasta?,0,0,0,0
30,NaN,NaN,The end!,0,0,0,0
31,ablaze,Birmingham,@bbcmtd Wholesale Markets ablaze http://t.co/l...,1,1,1,1
32,ablaze,Est. September 2012 - Bristol,We always try to bring the heavy. #metal #RT h...,0,1,1,0
33,ablaze,AFRICA,#AFRICANBAZE: Breaking news:Nigeria flag set a...,1,1,1,1
34,ablaze,"Philadelphia, PA",Crying out for more! Set me ablaze,0,1,1,1
35,ablaze,"London, UK",On plus side LOOK AT THE SKY LAST NIGHT IT WAS...,0,1,1,1
36,ablaze,Pretoria,@PhDSquares #mufc they've built so much hype a...,0,1,1,1


In [10]:
matched=train[train["keywordInText"]==True]
matched

,keyword,location,text,target,keywordDefined,locationDefined,keywordInText
31,ablaze,Birmingham,@bbcmtd Wholesale Markets ablaze http://t.co/l...,1,1,1,1
33,ablaze,AFRICA,#AFRICANBAZE: Breaking news:Nigeria flag set a...,1,1,1,1
34,ablaze,"Philadelphia, PA",Crying out for more! Set me ablaze,0,1,1,1
35,ablaze,"London, UK",On plus side LOOK AT THE SKY LAST NIGHT IT WAS...,0,1,1,1
36,ablaze,Pretoria,@PhDSquares #mufc they've built so much hype a...,0,1,1,1
...,...,...,...,...,...,...,...
7578,wrecked,NaN,@jt_ruff23 @cameronhacker and I wrecked you both,0,1,0,1
7579,wrecked,"Vancouver, Canada",Three days off from work and they've pretty mu...,0,1,1,1
7580,wrecked,London,#FX #forex #trading Cramer: Iger's 3 words tha...,0,1,1,1
7581,wrecked,Lincoln,@engineshed Great atmosphere at the British Li...,0,1,1,1


In [11]:
print(matched.shape[0])
print(train["keyword"].notna().sum())

5973
7552


In [12]:
import re 

STOPWORDS = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours',
    'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its', 'itself',
    'they', 'them', 'their', 'theirs', 'themselves', 'a', 'an', 'and', 'are', 'as', 'at',
    'be', 'by', 'for', 'from', 'has', 'have', 'he', 'in', 'is', 'it', 'of', 'on', 'or',
    'that', 'the', 'this', 'to', 'was', 'were', 'will', 'with', 'but', 'not', 'so', 'than',
    'then', 'thence', 'there', 'these', 'they', 'this', 'those', 'through', 'thus', 'very',
    'just', 'don', 'dont', 'should', 'shouldve', 'now', 'theres', 'wasnt', 'werent'
])

def clean_text(text):
    """Clean and preprocess text without NLTK"""
    if pd.isna(text):
        return ""
    
    text = text.lower() # Convert to lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Remove URLs
    text = re.sub(r'@(\w+)', r'\1', text) # Remove mentions (keep the word without @ or #)
    text = re.sub(r'#(\w+)', r'\1', text) # Remove hashtags (keep the word without @ or #)
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation and special characters
    text = re.sub(r'\d+', '', text) # Remove numbers
    text = ' '.join(text.split()) # Remove extra whitespace
    
    return text

def remove_stopwords(text):
    """Remove stopwords from text"""
    words = text.split()
    words = [w for w in words if w not in STOPWORDS]
    return ' '.join(words)

In [13]:
# Basic text features
train["text_length"] = train["text"].str.len().fillna(0)
train["word_count"] = train["text"].str.split().str.len().fillna(0)

# Additional features (no external libraries)
train["capital_count"] = train["text"].str.findall(r'[A-Z]').str.len()
train["exclamation_count"] = train["text"].str.count('!')
train["question_count"] = train["text"].str.count('\\?')
train["hashtag_count"] = train["text"].str.count('#')
train["mention_count"] = train["text"].str.count('@')

# apply preproc
train["clean_text"] = train["text"].apply(clean_text)
train["clean_text"] = train["clean_text"].apply(remove_stopwords)

In [14]:
test["keywordDefined"]=pd.Series([
  int(pd.notna(keyword)) for keyword in train["keyword"]
])
test["locationDefined"]=pd.Series([
  int(pd.notna(loc)) for loc in train["location"]
])
test["keywordInText"]=pd.Series([
	int(str(keyword).lower() in str(text).lower()) if keyword else False for keyword,text in zip(train["keyword"],train["text"])
])

# Text features for test
test["text_length"] = test["text"].str.len().fillna(0)
test["word_count"] = test["text"].str.split().str.len().fillna(0)
test["capital_count"] = test["text"].str.findall(r'[A-Z]').str.len()
test["exclamation_count"] = test["text"].str.count('!')
test["question_count"] = test["text"].str.count('\\?')
test["hashtag_count"] = test["text"].str.count('#')
test["mention_count"] = test["text"].str.count('@')

# Apply text preprocessing to test
test["clean_text"] = test["text"].apply(clean_text)
test["clean_text"] = test["clean_text"].apply(remove_stopwords)

In [15]:
train
# test

,keyword,location,text,target,keywordDefined,locationDefined,keywordInText,text_length,word_count,capital_count,exclamation_count,question_count,hashtag_count,mention_count,clean_text
0,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1,0,0,0,69,13,10,0,0,1,0,deeds reason earthquake may allah forgive us all
1,NaN,NaN,Forest fire near La Ronge Sask. Canada,1,0,0,0,38,7,5,0,0,0,0,forest fire near la ronge sask canada
2,NaN,NaN,All residents asked to 'shelter in place' are ...,1,0,0,0,133,22,2,0,0,0,0,all residents asked shelter place being notifi...
3,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1,0,0,0,65,8,1,0,0,1,0,people receive wildfires evacuation orders cal...
4,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1,0,0,0,88,16,3,0,0,2,0,got sent photo ruby alaska smoke wildfires pou...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7608,NaN,NaN,Two giant cranes holding a bridge collapse int...,1,0,0,0,83,11,7,0,0,0,0,two giant cranes holding bridge collapse into ...
7609,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1,0,0,0,125,20,6,0,0,0,2,aria_ahrary thetawniest out control wild fires...
7610,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1,0,0,0,65,8,10,0,1,0,0,m utckm s volcano hawaii
7611,NaN,NaN,Police investigating after an e-bike collided ...,1,0,0,0,137,19,4,0,0,0,0,police investigating after ebike collided car ...


# prep

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF features (deterministic - no random components)
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,           # Limit features for manageable size
    ngram_range=(1, 2),         # Unigrams and bigrams
    stop_words=list(STOPWORDS),
)

# Fit on training data, transform both
X_train_tfidf = tfidf_vectorizer.fit_transform(train["clean_text"])
X_test_tfidf = tfidf_vectorizer.transform(test["clean_text"])

print(f"TF-IDF features shape: {X_train_tfidf.shape}")
print(f"Sample feature names: {tfidf_vectorizer.get_feature_names_out()[:20]}")

TF-IDF features shape: (7613, 5000)
Sample feature names: ['__' 'aba' 'aba woman' 'abandon' 'abandoned' 'abandoned aircraft'
 'abbswinston' 'abbswinston zionist' 'abc' 'abc news' 'abcnews'
 'abcnews obama' 'abe' 'ability' 'ablaze' 'able' 'about' 'about cable'
 'about explode' 'about how']


In [17]:
from scipy.sparse import hstack, csr_matrix

# Select numeric features
numeric_features = [
    "keywordDefined", "locationDefined", "keywordInText",
    "text_length", "word_count", "capital_count",
    "exclamation_count", "question_count", "hashtag_count", 
    "mention_count"
]

# Extract numeric features and convert to sparse matrices
X_train_numeric = csr_matrix(train[numeric_features].values.astype(np.float32))
X_test_numeric = csr_matrix(test[numeric_features].values.astype(np.float32))

# Combine TF-IDF and numeric features
X_train_combined = hstack([X_train_tfidf, X_train_numeric])
X_test_combined = hstack([X_test_tfidf, X_test_numeric])

print(f"Combined training features shape: {X_train_combined.shape}")
print(f"Combined test features shape: {X_test_combined.shape}")

# Target variable
y_train = train["target"].values

Combined training features shape: (7613, 5010)
Combined test features shape: (3263, 5010)


In [18]:
from sklearn.model_selection import train_test_split

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_combined, y_train, 
    test_size=0.2, 
    random_state=RANDOMSEED,
    stratify=y_train  # Maintain class distribution
)

print(f"Training set: {X_train_split.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Class distribution - Train: {np.bincount(y_train_split)}")
print(f"Class distribution - Val: {np.bincount(y_val)}")

Training set: 6090 samples
Validation set: 1523 samples
Class distribution - Train: [3473 2617]
Class distribution - Val: [869 654]


# training

we'll be ensemblemaxxing for this problem

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import xgboost as xgb

# Initialize individual models (all deterministic)
logistic_model = LogisticRegression(
    C=1.0,
    max_iter=5000,
    random_state=RANDOMSEED,
    solver='liblinear',  # Deterministic solver
    class_weight='balanced',
    n_jobs=None  # Single-threaded for determinism
)
xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=3,
    learning_rate=0.01,
    random_state=RANDOMSEED,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=1,  # Single-threaded for determinism
    tree_method='hist'  # Deterministic tree method
)
bnb_model=BernoulliNB(
  binarize=0.5,
)

In [20]:
ensem = VotingClassifier(
    estimators=[
        ('lr', logistic_model),
        ('xgb', xgb_model),
        ('bnb',bnb_model),
    ],
    voting='hard',
    verbose=True,
)

In [21]:
ensem.fit(X_train_split, y_train_split)

[Voting] ....................... (1 of 3) Processing lr, total=   0.1s


c:\Users\longnuub\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:57:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[Voting] ...................... (2 of 3) Processing xgb, total=   3.2s
[Voting] ...................... (3 of 3) Processing bnb, total=   0.0s


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('lr', ...), ('xgb', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'hard'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001


In [22]:
y_pred_ensemble = ensem.predict(X_val)

# predictions

predict

In [23]:
preds = ensem.predict(X_test_combined)

In [24]:
print(f"Unique prediction values: {np.unique(preds)}")

Unique prediction values: [0 1]


In [25]:
subs_df=pd.DataFrame({
  "ID":test["id"],
  "Target":preds,
})

In [26]:
subs_df.head()
# subs_df.shape

,ID,Target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1


In [27]:
subs_df.to_csv("submission.csv",index=False)